In [ ]:
#需要使用(venv)python 3.9.6
import sqlite3

# 快速測試連接
try:
    conn = sqlite3.connect(':memory:')  # 記憶體資料庫
    print("SQLite連接成功！")
    conn.close()
except sqlite3.Error as e:
    print(f"連接失敗: {e}")

In [3]:
%pip install ipython-sql

175.08s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Note: you may need to restart the kernel to use updated packages.


In [3]:
import sqlite3

# 假設您的 .ipynb 檔案在 Hello-World/ 底下
# 直接使用相對於該檔案的路徑即可
db_path = 'demo.sqlite'

try:
    with sqlite3.connect(db_path) as conn:
        print(f"成功連線到資料庫: {db_path}")
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        print("資料庫中的資料表：", cursor.fetchall())

except sqlite3.Error as e:
    print(f"資料庫發生錯誤: {e}")


成功連線到資料庫: demo.sqlite
資料庫中的資料表： [('products',), ('sqlite_sequence',)]


In [4]:
import sqlite3
from pathlib import Path

# 取得當前工作目錄 (也就是 .ipynb 檔案所在的目錄)
project_root = Path.cwd()

# 組合出完整的資料庫路徑
db_path = project_root / 'demo.sqlite'

try:
    with sqlite3.connect(db_path) as conn:
        print(f"成功連線到資料庫: {db_path}")
        cursor = conn.cursor()
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
        print("資料庫中的資料表：", cursor.fetchall())

except sqlite3.Error as e:
    print(f"資料庫發生錯誤: {e}")


成功連線到資料庫: /Users/hachi/VSCode-Training/Hello-World/SQL/demo.sqlite
資料庫中的資料表： [('products',), ('sqlite_sequence',)]


In [5]:
import sqlite3
import pandas as pd
from pathlib import Path

# --- 在 Jupyter Notebook 中連線與查詢的正確方式 ---

# 因為您的 .ipynb 檔案和 demo.sqlite 在同一個資料夾 (SQL/)
# 所以我們可以直接使用檔案名稱作為相對路徑
db_path = 'demo.sqlite'

# 為了方便除錯，我們印出計算後的絕對路徑
# 這樣可以確認程式是否真的找到了正確的位置
print(f"嘗試連線到資料庫: {Path(db_path).resolve()}")

try:
    # 使用 'with' 陳述式可以確保連線在使用後會被安全地關閉
    with sqlite3.connect(db_path) as conn:
        print("資料庫連線成功！")

        # 使用 pandas 讀取資料表，這是最方便的方式
        query = "SELECT * FROM products;"
        df = pd.read_sql_query(query, conn)

        # 在 Jupyter 中使用 display() 可以將 DataFrame 以漂亮的表格顯示出來
        print("\n'products' 資料表內容：")
        display(df)

except sqlite3.Error as e:
    print(f"資料庫發生錯誤: {e}")
    print("\n請確認：")
    print(f"1. 資料庫檔案是否真的存在於 '{Path(db_path).resolve()}'？")
    print("2. 檔案或資料夾名稱是否拼寫錯誤？")



嘗試連線到資料庫: /Users/hachi/VSCode-Training/Hello-World/SQL/demo.sqlite
資料庫連線成功！

'products' 資料表內容：


,id,name,price
0,1,Laptop,1500.0
1,2,Keyboard,100.0
2,3,Mouse,350.0
3,4,Monitor,4200.0


In [1]:
import sqlite3

# 建立或連接 demo.sqlite 資料庫
conn = sqlite3.connect('demo.sqlite')
cursor = conn.cursor()

# 啟用 foreign key 約束（SQLite 需要明確啟用）
cursor.execute("PRAGMA foreign_keys = ON;")

# 第一步：建立一個分類表 categories（讓 products 有外鍵可參照）
cursor.execute("""
CREATE TABLE IF NOT EXISTS categories (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL
);
""")

# 第二步：建立 products 表，包含外鍵 category_id
cursor.execute("""
CREATE TABLE IF NOT EXISTS products (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    price REAL NOT NULL,
    category_id INTEGER,
    FOREIGN KEY (category_id) REFERENCES categories(id)
);
""")

# 關閉資料庫連線
conn.commit()
conn.close()
